# Notebook 01 — Collecte météo
**Pipeline :** Top 35 villes FR → coordonnées GPS (Nominatim) → prévisions 7 jours (OpenWeatherMap) → score météo → `cities.csv`

## 0. Imports

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import plotly.express as px
from pathlib import Path
from dotenv import load_dotenv

# Rendre src/ importable depuis notebooks/
sys.path.append(str(Path().resolve().parent))

from src.nominatim import get_all_coordinates
from src.weather import collect_weather_data, compute_weather_score

load_dotenv(dotenv_path=Path().resolve().parent / ".env")
OWM_API_KEY = os.getenv("OWM_API_KEY")

assert OWM_API_KEY, "OWM_API_KEY manquante dans le fichier .env !"
print("Clé OWM : OK")

Clé OWM : OK


## 1. Liste des 35 villes

In [3]:
CITIES = [
"Mont Saint Michel","St Malo","Bayeux","Le Havre","Rouen","Paris",
"Amiens","Lille","Strasbourg","Chateau du Haut Koenigsbourg","Colmar",
"Eguisheim","Besancon","Dijon","Annecy","Grenoble","Lyon",
"Gorges du Verdon","Bormes les Mimosas","Cassis","Marseille",
"Aix en Provence","Avignon","Uzes","Nimes","Aigues Mortes",
"Saintes Maries de la mer","Collioure","Carcassonne","Ariege",
"Toulouse","Montauban","Biarritz","Bayonne","La Rochelle"
]
print(f"{len(CITIES)} villes")

35 villes


## 2. Coordonnées GPS via Nominatim

In [4]:
# Cellule 2 — Coordonnées GPS
from src.nominatim import get_all_coordinates

df_cities = get_all_coordinates(CITIES)
print(f"\nShape : {df_cities.shape}")
df_cities.head()

  0 villes en cache | 35 à récupérer via Nominatim
    API → Mont Saint Michel         lat=48.6359541, lon=-1.51146
    API → St Malo                   lat=48.649518, lon=-2.0260409
    API → Bayeux                    lat=49.2764624, lon=-0.7024738
    API → Le Havre                  lat=49.4938975, lon=0.1079732
    API → Rouen                     lat=49.4404591, lon=1.0939658
    API → Paris                     lat=48.8588897, lon=2.320041
    API → Amiens                    lat=49.8941708, lon=2.2956951
    API → Lille                     lat=50.6365654, lon=3.0635282
    API → Strasbourg                lat=48.584614, lon=7.7507127
    API → Chateau du Haut Koenigsbourg lat=48.249382, lon=7.3439412
    API → Colmar                    lat=48.0777517, lon=7.3579641
    API → Eguisheim                 lat=48.0447968, lon=7.3079618
    API → Besancon                  lat=47.2380222, lon=6.0243622
    API → Dijon                     lat=47.3215806, lon=5.0414701
    API → Annecy         

/home/ibraba/tourisme-france/src/nominatim.py:84: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache  = pd.concat([cache, df_new], ignore_index=True)


,city_id,city,lat,lon
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


## 3. Prévisions météo via OpenWeatherMap

In [9]:
df_weather = collect_weather_data(df_cities, OWM_API_KEY, delay=0.5)

print(f"Shape : {df_weather.shape}  ({len(df_cities)} villes × 7 jours)")
df_weather.head()

  Pas de cache — appel OWM pour 35 villes...
    OK Mont Saint Michel
    OK St Malo
    OK Bayeux
    OK Le Havre
    OK Rouen
    OK Paris
    OK Amiens
    OK Lille
    OK Strasbourg
    OK Chateau du Haut Koenigsbourg
    OK Colmar
    OK Eguisheim
    OK Besancon
    OK Dijon
    OK Annecy
    OK Grenoble
    OK Lyon
    OK Gorges du Verdon
    OK Bormes les Mimosas
    OK Cassis
    OK Marseille
    OK Aix en Provence
    OK Avignon
    OK Uzes
    OK Nimes
    OK Aigues Mortes
    OK Saintes Maries de la mer
    OK Collioure
    OK Carcassonne
    OK Ariege
    OK Toulouse
    OK Montauban
    OK Biarritz
    OK Bayonne
    OK La Rochelle
  Cache sauvegardé : weather_cache_2026-04-30.csv (280 lignes)
Shape : (280, 13)  (35 villes × 7 jours)


,city_id,city,lat,lon,date,temp_day,temp_min,temp_max,humidity,pop,rain_mm,weather_main,weather_desc
0,1,Mont Saint Michel,48.635954,-1.51146,2026-04-30,18.98,12.43,18.98,63,0.99,1.37,Rain,light rain
1,1,Mont Saint Michel,48.635954,-1.51146,2026-05-01,16.17,12.12,18.34,81,0.54,0.00,Clouds,overcast clouds
2,1,Mont Saint Michel,48.635954,-1.51146,2026-05-02,17.51,12.13,18.16,74,1.00,8.80,Rain,moderate rain
3,1,Mont Saint Michel,48.635954,-1.51146,2026-05-03,15.38,10.22,16.79,83,1.00,5.26,Rain,light rain
4,1,Mont Saint Michel,48.635954,-1.51146,2026-05-04,14.59,8.64,14.59,79,0.20,0.37,Rain,light rain


## 4. Score météo par ville

Formule : `score = temp_moy - (pop_moy × 10) - (rain_total × 0.5)`  
Tu peux modifier les coefficients dans `src/weather.py`.

In [10]:
df_score = compute_weather_score(df_weather)

print("Top 10 destinations :")
df_score[["rank", "city", "temp_moy", "pop_moy", "rain_total", "weather_score"]].head(10)

Top 10 destinations :


,rank,city,temp_moy,pop_moy,rain_total,weather_score
0,1,Collioure,20.60500,0.29500,3.12,16.10
1,2,Carcassonne,19.63250,0.45875,6.65,11.72
2,3,Strasbourg,19.00375,0.46125,11.51,8.64
3,4,Lille,18.01625,0.57000,11.58,6.53
4,5,Colmar,18.98625,0.60000,13.23,6.37
5,6,Eguisheim,18.80625,0.58875,13.90,5.97
6,7,Chateau du Haut Koenigsbourg,16.05250,0.56000,11.37,4.77
7,8,Saintes Maries de la mer,17.41750,0.41750,18.15,4.17
8,9,Toulouse,19.69000,0.79375,15.61,3.95
9,10,Besancon,17.05500,0.46875,18.83,2.95


## 5. Carte Plotly — Top 5 destinations

In [11]:
top5 = df_score.head(5)

# Normaliser le score pour l'utiliser comme taille et ajouter +2 pour éviter 0
score_min = df_score["weather_score"].min()
df_score["size_plot"] = df_score["weather_score"] - score_min + 2  

fig = px.scatter_mapbox(
    df_score,
    lat="lat", lon="lon",
    hover_name="city",
    hover_data={"weather_score": True, "temp_moy": ":.1f", "rain_total": ":.1f", "size_plot": False},
    color="weather_score",
    size="size_plot",                  
    color_continuous_scale="RdYlGn",
    size_max=20,
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    mapbox_style="carto-positron",
    title="Top 35 villes françaises — score météo 7 jours"
)

for _, r in top5.iterrows():
    fig.add_annotation(
        x=r["lon"], y=r["lat"],
        text=f"#{int(r['rank'])} {r['city']}",
        showarrow=False,
        font=dict(size=11, color="darkgreen"),
        bgcolor="white",
        opacity=0.85
    )

fig.show()

## 6. Sauvegarde

In [8]:
out = Path().resolve().parent / "data" / "raw" / "cities.csv"
df_score.to_csv(out, index=False)
print(f"Sauvegardé : {out}")
print(f"Colonnes   : {list(df_score.columns)}")
print(f"Lignes     : {len(df_score)}")

Sauvegardé : /home/ibraba/tourisme-france/data/raw/cities.csv
Colonnes   : ['city_id', 'city', 'lat', 'lon', 'temp_moy', 'temp_max', 'pop_moy', 'rain_total', 'humidity', 'weather_score', 'rank', 'size_plot']
Lignes     : 35
